# آموزش ترکیبی Qwen MoE: آرشیو معلم + دادهٔ فارسی + پایلوت

این نوت‌بوک آرشیو کامل پرامپت/پاسخ معلم را روی Google Drive نگه می‌دارد؛ آموزش فقط از یک زیرمجموعهٔ محدود و نسخه‌دار استفاده می‌کند. آرشیو منبع اصلی است و هرگز با فایل‌های انتخابی آموزش جایگزین نمی‌شود.

**محدودیت علمی:** مسیر فعلی تقطیر activation در FFN است؛ تقطیر مستقیم logits یا SFT کامل روی پاسخ‌ها هنوز پیاده‌سازی نشده است. پاسخ‌های ذخیره‌شده برای بازآموزی/ارزیابی آینده قابل استفاده‌اند.

**مدل معلم فعلی ریپو:** `Qwen/Qwen3.5-2B`. قبل از آموزش نهایی، شناسهٔ مدل هدف باید یک‌دست شود.این نوت‌بوک برای اجرای مرحله‌ای در Google Colab طراحی شده است و دو مجموعهٔ جدا می‌سازد: حداکثر ۵۰۰٬۰۰۰ توکن متن/کتاب فارسی و حداکثر ۵۰۰٬۰۰۰ توکن ادامهٔ تولیدشده توسط مدل معلم. خروجی‌ها روی Google Drive ذخیره می‌شوند و تولید معلم قابلیت ادامه پس از قطع runtime دارد.

**محدودیت علمی:** اسکریپت فعلی پروژه خروجی FFN معلم را روی متن‌ها تقطیر می‌کند؛ این نسخه distillation مستقیم logits یا آموزش کامل SFT روی پاسخ‌ها نیست. دادهٔ تولیدشدهٔ معلم، متن آموزشی اضافه برای تقطیر activation است. این آزمایش به‌تنهایی کیفیت استدلال یا مدل کامل را تضمین نمی‌کند.

**مدل معلم فعلی ریپو:** `Qwen/Qwen3.5-2B`. اگر هدف نهایی حتماً `Instruct` یا `Base` دیگری است، قبل از اجرای این نوت‌بوک باید شناسهٔ مدل را یک‌دست کنیم. آموزش کامل ۲۴ لایه در این نوت‌بوک عمداً خودکار اجرا نمی‌شود.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, subprocess, shutil, time, hashlib
WORK = Path('/content/drive/MyDrive/qwen-moe-combined-distillation')
WORK.mkdir(parents=True, exist_ok=True)
MODEL_ID = 'Qwen/Qwen3.5-2B'
RAW_TOKEN_BUDGET = 500_000
TEACHER_TOKEN_BUDGET = 500_000
print('Persistent work directory:', WORK)
print('Teacher:', MODEL_ID)


In [ ]:
if not Path('/content/Qwen-MoE-0.4B-').exists():
    subprocess.run(['git','clone','https://github.com/samanramezani1377-hub/Qwen-MoE-0.4B-.git','/content/Qwen-MoE-0.4B-'],check=True)
%cd /content/Qwen-MoE-0.4B-
!pip -q install -U transformers accelerate safetensors datasets sentencepiece
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('برای تولید ۵۰۰هزار توکن معلم، GPU لازم است. از Runtime > Change runtime type، GPU را انتخاب کن.')
print('GPU:', torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))


## مرحلهٔ ۱ — گردآوری متن واقعی فارسی

منبع‌ها از اسکریپت موجود پروژه انتخاب می‌شوند؛ مجوز و منشأ در manifest ثبت می‌شود. چون شمار سند با شمار توکن برابر نیست، ابتدا corpus بزرگ‌تری می‌گیریم و بعد با tokenizer خود مدل سقف ۵۰۰هزار توکن را اعمال می‌کنیم. اگر منبع در دسترس نباشد، کار با خطا متوقف می‌شود؛ متن ساختگی جایگزین نمی‌شود.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
corpus_path = WORK / 'fa_raw_corpus.txt'
corpus_manifest = WORK / 'fa_raw_corpus.manifest.json'
manifest_status = None
if corpus_manifest.exists():
    try: manifest_status = json.loads(corpus_manifest.read_text(encoding='utf-8')).get('status')
    except Exception: manifest_status = None
existing_count = len([x for x in corpus_path.read_text(encoding='utf-8').splitlines() if x.strip()]) if corpus_path.exists() else 0
if not corpus_path.exists() or not corpus_manifest.exists() or manifest_status != 'success' or existing_count < 6000:
    subprocess.run([sys.executable,'scripts/collect_persian_corpus.py','--count','6000','--min-chars','160','--max-chars','3000','--max-scan','100000','--output',str(corpus_path),'--manifest-output',str(corpus_manifest)],check=True)
docs = [x.strip() for x in corpus_path.read_text(encoding='utf-8').splitlines() if x.strip()]
print('Documents:', len(docs), '| manifest:', json.loads(corpus_manifest.read_text(encoding='utf-8')).get('dataset'))
if len(docs) < 100: raise RuntimeError('کمتر از ۱۰۰ سند واقعی موجود است؛ داده برای آزمایش ترکیبی کافی نیست.')


In [ ]:
def trim_docs_to_token_budget(documents, budget, tok):
    chosen=[]; total=0
    for doc in documents:
        ids = tok.encode(doc, add_special_tokens=False)
        if not ids: continue
        remaining = budget-total
        if remaining <= 0: break
        ids = ids[:remaining]
        piece = tok.decode(ids, skip_special_tokens=True)
        if piece.strip(): chosen.append(piece); total += len(tok.encode(piece, add_special_tokens=False))
    # Tokenization round-trip can slightly change counts; enforce the limit conservatively.
    out=[]; total=0
    for doc in chosen:
        ids=tok.encode(doc, add_special_tokens=False); remaining=budget-total
        if remaining<=0: break
        if len(ids)>remaining: ids=ids[:remaining]
        piece=tok.decode(ids,skip_special_tokens=True)
        if piece.strip(): out.append(piece); total += len(tok.encode(piece,add_special_tokens=False))
    return out, total
raw_docs, raw_tokens = trim_docs_to_token_budget(docs, RAW_TOKEN_BUDGET, tokenizer)
raw_train_path = WORK / 'raw_fa_500k_tokens.txt'
raw_train_path.write_text('\n'.join(raw_docs)+'\n', encoding='utf-8')
(WORK/'raw_fa_token_manifest.json').write_text(json.dumps({'target_token_budget':RAW_TOKEN_BUDGET,'token_count_after_roundtrip':raw_tokens,'documents':len(raw_docs),'tokenizer_model':MODEL_ID,'source_manifest':str(corpus_manifest)},ensure_ascii=False,indent=2),encoding='utf-8')
print('Raw corpus token count:', raw_tokens, '/', RAW_TOKEN_BUDGET, '| docs:', len(raw_docs))
if raw_tokens < 400_000: raise RuntimeError(f'متن کافی جمع نشده: {raw_tokens} توکن. منبع/سقف جمع‌آوری را بررسی کن؛ آموزش را با دادهٔ ناقص شروع نکن.')


## مرحلهٔ ۲ — تولید معلم و آرشیو ماندگار

`teacher_generated.jsonl` آرشیو کامل و مرجع است: پرامپت اصلی و مؤثر، پاسخ، token IDs، نسخه و تنظیمات تولید. هر نمونه جداگانه append و flush می‌شود. اگر Colab قطع شود، اجرای مجدد از رکوردهای ذخیره‌شده ادامه می‌دهد. فایل‌های `teacher_train_subset.jsonl` و `teacher_eval_holdout.jsonl` فقط نماهای انتخابی هستند و می‌توان آن‌ها را دوباره ساخت؛ آرشیو اصلی حفظ می‌شود.

به‌طور پیش‌فرض حداکثر ۲٬۰۰۰ نمونه از بخش train برای پایلوت صادر می‌شود؛ همهٔ آرشیو مستقیماً وارد آموزش نمی‌شود. مجموعهٔ eval از train جدا نگه داشته می‌شود.


In [ ]:
from transformers import AutoModelForCausalLM
import torch, json, time, gc, hashlib
teacher_jsonl = WORK / 'teacher_generated.jsonl'
teacher_text_path = WORK / 'teacher_generated_500k.txt'
teacher_manifest_path = WORK / 'teacher_generation_manifest.json'
archive_manifest_path = WORK / 'teacher_archive_manifest.json'
teacher_train_jsonl = WORK / 'teacher_train_subset.jsonl'
teacher_eval_jsonl = WORK / 'teacher_eval_holdout.jsonl'
teacher_eval_text_path = WORK / 'teacher_eval_responses.txt'
GEN_SEED = 20261025
torch.manual_seed(GEN_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(GEN_SEED)
if not teacher_jsonl.exists(): teacher_jsonl.write_text('', encoding='utf-8')
# Recover only an incomplete final JSONL record; quarantine its bytes before truncating.
raw_lines = teacher_jsonl.read_bytes().splitlines(keepends=True)
while raw_lines:
    last = raw_lines[-1]
    if not last.strip():
        raw_lines.pop()
        continue
    try:
        json.loads(last.decode('utf-8'))
        break
    except Exception:
        if any(line.strip() for line in raw_lines[:-1]):
            try:
                json.loads(raw_lines[-2].decode('utf-8')) if raw_lines[-2].strip() else None
            except Exception as exc:
                raise RuntimeError('A malformed non-final record was found in the teacher archive; preserve it and inspect manually before continuing.') from exc
        quarantine = teacher_jsonl.with_suffix('.jsonl.incomplete_tail')
        with quarantine.open('ab') as qf:
            qf.write(last); qf.flush(); os.fsync(qf.fileno())
        raw_lines.pop()
        break
teacher_jsonl.write_bytes(b''.join(raw_lines))
existing = [json.loads(x) for x in teacher_jsonl.read_text(encoding='utf-8').splitlines() if x.strip()]
def response_of(r): return r.get('response_text', r.get('text', ''))
generated_texts = [response_of(r) for r in existing if response_of(r)]
def count_tokens(texts): return sum(len(tokenizer.encode(t, add_special_tokens=False)) for t in texts)
already = sum(len(tokenizer.encode(t, add_special_tokens=False)) for t in generated_texts)
print('Resuming teacher generation from', already, 'tokens')
if already < TEACHER_TOKEN_BUDGET:
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    teacher = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map='auto', low_cpu_mem_usage=True)
    teacher.eval()
    teacher_revision_saved = getattr(teacher.config, '_commit_hash', None)
    prompt_pool = raw_docs
    if not prompt_pool: raise RuntimeError('No prompts available')
    idx = len(existing); since_save = 0; started = time.time()
    while already < TEACHER_TOKEN_BUDGET:
        prompt_doc_index = idx % len(prompt_pool)
        prompt_text = prompt_pool[prompt_doc_index][:1800]
        pids = tokenizer.encode(prompt_text, return_tensors='pt', truncation=True, max_length=384).to(teacher.device)
        prompt_token_ids = pids[0].detach().cpu().tolist()
        effective_prompt_text = tokenizer.decode(prompt_token_ids, skip_special_tokens=False)
        allowance = min(128, TEACHER_TOKEN_BUDGET - already)
        generation_config = {'do_sample':True,'temperature':0.7,'top_p':0.9,'repetition_penalty':1.05,'max_new_tokens':allowance,'use_cache':True,'seed':GEN_SEED}
        with torch.inference_mode():
            out = teacher.generate(input_ids=pids, max_new_tokens=allowance, do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.05, pad_token_id=tokenizer.eos_token_id, use_cache=True)
        response_token_ids = out[0, pids.shape[1]:].detach().cpu().tolist()
        response_text = ' '.join(tokenizer.decode(response_token_ids, skip_special_tokens=True).split())
        if response_text:
            rec={
                'schema_version':2,
                'sample_id':hashlib.sha256((str(idx)+json.dumps(prompt_token_ids)+json.dumps(response_token_ids)).encode()).hexdigest(),
                'index':idx,
                'prompt_document_index':prompt_doc_index,
                'prompt_source_text':prompt_text,
                'prompt_text_as_tokenized':effective_prompt_text,
                'prompt_token_ids':prompt_token_ids,
                'response_text':response_text,
                'response_token_ids':response_token_ids,
                'teacher_model_id':MODEL_ID,
                'teacher_model_revision':teacher_revision_saved,
                'tokenizer_model_id':MODEL_ID,
                'generation_config':generation_config,
                'dtype':str(dtype),
                'created_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())
            }
            # Append and flush every sample: a runtime disconnect should lose at most the in-flight generation.
            with teacher_jsonl.open('a',encoding='utf-8') as f:
                f.write(json.dumps(rec,ensure_ascii=False)+'\n'); f.flush(); os.fsync(f.fileno())
            generated_texts.append(response_text)
            already += len(response_token_ids)
        idx += 1; since_save += 1
        if since_save >= 25:
            print(f'Generated {already:,}/{TEACHER_TOKEN_BUDGET:,} tokens | examples={len(generated_texts)} | elapsed_min={(time.time()-started)/60:.1f}', flush=True)
            since_save=0
        if idx % 100 == 0: torch.cuda.empty_cache()
    del teacher; gc.collect(); torch.cuda.empty_cache()
# The full JSONL remains the source of truth. Export separate, reproducible train/eval views.
archive_cmd = [
    sys.executable, 'scripts/prepare_teacher_archive.py',
    '--archive', str(teacher_jsonl),
    '--manifest', str(archive_manifest_path),
    '--train-jsonl', str(teacher_train_jsonl),
    '--eval-jsonl', str(teacher_eval_jsonl),
    '--train-text', str(teacher_text_path),
    '--eval-text', str(teacher_eval_text_path),
    '--eval-fraction', '0.10',
    '--max-train-samples', '2000',
    '--repair-trailing-line'
]
subprocess.run(archive_cmd, check=True)
archive_rows = [json.loads(x) for x in teacher_jsonl.read_text(encoding='utf-8').splitlines() if x.strip()]
archive_tokens = sum(len(r.get('response_token_ids', [])) for r in archive_rows)
teacher_revision = next((r.get('teacher_model_revision') for r in archive_rows if r.get('teacher_model_revision')), None)
teacher_manifest = {
    'status': 'success' if archive_tokens >= 400_000 else 'partial',
    'target_tokens': TEACHER_TOKEN_BUDGET,
    'archived_response_tokens_from_saved_token_ids': archive_tokens,
    'archive_records': len(archive_rows),
    'teacher_model_id': MODEL_ID,
    'model_revision': teacher_revision,
    'archive_jsonl': str(teacher_jsonl),
    'archive_manifest': str(archive_manifest_path),
    'train_subset_jsonl': str(teacher_train_jsonl),
    'eval_holdout_jsonl': str(teacher_eval_jsonl),
    'train_response_text_file': str(teacher_text_path),
    'eval_response_text_file': str(teacher_eval_text_path),
    'note': 'Archive is retained; training and evaluation files are deterministic derived views.'
}
teacher_manifest_path.write_text(json.dumps(teacher_manifest, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print('Full prompt-response archive (retain this):', teacher_jsonl)
print('Selected train response text:', teacher_text_path)
print('Held-out eval archive:', teacher_eval_jsonl)
print('Archive tokens:', archive_tokens, '| selected train examples:', len([x for x in teacher_train_jsonl.read_text(encoding='utf-8').splitlines() if x.strip()]))


## مرحلهٔ ۳ — ترکیب دو مجموعه و تفکیک ارزیابی

از هر دو مجموعه متن واقعی و متن تولیدشدهٔ معلم استفاده می‌شود. برای جلوگیری از نشت، اسناد قبل از آموزش deduplicate و به train/eval تقسیم می‌شوند. اسکریپت تقطیر خودش نیز یک تقسیم سندی داخلی دارد. اگر شمار توکن خروجی معلم کمتر از هدف باشد، سلول متوقف می‌شود.

In [ ]:
# Use only the capped train subset of teacher responses; the complete archive is never fed wholesale to training.
teacher_texts = [x.strip() for x in teacher_text_path.read_text(encoding='utf-8').splitlines() if x.strip()]
teacher_tokens = count_tokens(teacher_texts)
teacher_manifest = json.loads(teacher_manifest_path.read_text(encoding='utf-8'))
archive_tokens = int(teacher_manifest.get('archived_response_tokens_from_saved_token_ids', 0))
if archive_tokens < 400_000:
    raise RuntimeError(f'آرشیو معلم هنوز کامل نیست: {archive_tokens} توکن؛ سلول تولید را دوباره اجرا کن.')
if not teacher_texts:
    raise RuntimeError('زیرمجموعهٔ انتخابی train خالی است؛ آرشیو و manifest را بررسی کن.')
combined_path = WORK / 'combined_raw_plus_teacher.txt'
combined_docs = raw_docs + teacher_texts
seen=set(); unique=[]
for d in combined_docs:
    h=hashlib.sha256(d.encode('utf-8')).hexdigest()
    if h not in seen: seen.add(h); unique.append(d)
combined_path.write_text('\n'.join(unique)+'\n',encoding='utf-8')
subprocess.run([sys.executable,'scripts/split_persian_corpus.py','--input',str(combined_path),'--train-output',str(WORK/'combined_train.txt'),'--eval-output',str(WORK/'combined_eval.txt'),'--manifest-output',str(WORK/'combined_split_manifest.json'),'--eval-fraction','0.10','--seed','20261025'],check=True)
train_tokens=count_tokens([x for x in (WORK/'combined_train.txt').read_text(encoding='utf-8').splitlines() if x.strip()])
eval_tokens=count_tokens([x for x in (WORK/'combined_eval.txt').read_text(encoding='utf-8').splitlines() if x.strip()])
summary={
    'raw_tokens':raw_tokens,
    'teacher_archive_response_tokens':archive_tokens,
    'teacher_archive_records':teacher_manifest.get('archive_records'),
    'teacher_train_subset_records':len(teacher_texts),
    'teacher_eval_holdout_jsonl':str(teacher_eval_jsonl),
    'combined_unique_documents':len(unique),
    'combined_train_tokens':train_tokens,
    'combined_eval_tokens':eval_tokens,
    'teacher_model':MODEL_ID,
    'archive_manifest':str(archive_manifest_path),
    'policy':'Full teacher archive retained; only capped train subset is mixed into the training corpus.'
}
(WORK/'combined_data_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))


## مرحلهٔ ۴ — پایلوت تقطیر FFN لایهٔ صفر

این اجرا هنوز فقط یک FFN را آموزش می‌دهد؛ ۲۴ لایه اجرا نمی‌شوند. تنظیمات اولیه ۱۰ epoch، طول ۲۵۶ و ۳۲ expert با Top-2 است. اگر حافظه تمام شد، ابتدا طول را به ۱۲۸ و سپس batch را کم کن. معیارهای بازسازی به‌تنهایی برای پذیرش کافی نیستند؛ پس از آن باید ارزیابی generation و perplexity جداگانه اضافه/اجرا شود.

In [ ]:
CHECKPOINT_DIR = WORK / 'layer0_checkpoint'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
cmd=[sys.executable,'scripts/distill_all_qwen_ffn_layers.py','--model',MODEL_ID,'--text-file',str(WORK/'combined_train.txt'),'--output-dir',str(CHECKPOINT_DIR),'--num-experts','32','--top-k','2','--samples','2000','--epochs','5','--max-length','128','--batch-size','1','--train-batch-size','128','--router-pretrain-tokens','2048','--balance-loss-weight','0.01','--eval-fraction','0.1','--seed','20261025','--device','cuda','--start-layer','0','--end-layer','0']
print('Running:', ' '.join(cmd))
with (WORK/'layer0_training.log').open('w',encoding='utf-8') as logf:
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        print(line,end='')
        logf.write(line); logf.flush()
    code=proc.wait()
if code != 0: raise RuntimeError(f'Layer-0 distillation failed with exit code {code}. See {WORK}/layer0_training.log')
print('Pilot finished. No other layer was trained.')


In [ ]:
checkpoint_path=CHECKPOINT_DIR/'layer_00_top2_32experts.pt'
if not checkpoint_path.exists(): raise FileNotFoundError(checkpoint_path)
ckpt=torch.load(checkpoint_path,map_location='cpu',weights_only=False)
keys=['model_id','teacher_revision','layer','tokens_captured','train_tokens','eval_tokens','train_documents','eval_documents','eval_mse','eval_relative_error','eval_cosine_similarity','dead_experts_on_eval','router_entropy_nats','balance_loss_weight']
report={k:ckpt.get(k) for k in keys}; report['checkpoint']=str(checkpoint_path); report['data_summary']=summary
(WORK/'layer0_metrics_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print(json.dumps(report,ensure_ascii=False,indent=2,default=str))
print('\nاین گزارش فقط خطای بازسازی FFN را نشان می‌دهد. قبل از آموزش لایه‌های دیگر باید کیفیت تولید، perplexity، توافق توکن بعدی و آزمون‌های فارسی/استدلال بررسی شوند.')


## معیار توقف و ادامه

- اگر تولید معلم ناقص بود، سلول تولید را دوباره اجرا کن؛ از JSONL موجود ادامه می‌دهد.
- اگر خطای نسبی بازسازی بالا یا cosine پایین است، سراغ ۲۴ لایه نرو؛ روش expert/router باید اصلاح شود.
- اگر معیارهای لایهٔ صفر خوب بود، باز هم ابتدا چند لایهٔ نماینده و آزمون سرتاسری را اجرا کن.
- خروجی این نوت‌بوک مدل کامل Transformers، GGUF یا مدل آمادهٔ Android نیست.
- این نوت‌بوک هیچ سرویس یا GPU پولی را فعال نمی‌کند؛ محدودیت زمان و سهمیهٔ Colab رایگان ممکن است اجرای تولید ۵۰۰هزار توکن را چند جلسه‌ای کند.